# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [2]:
#%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

### Data Preparation

In [3]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [4]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: gchhablani/bert-base-cased-finetuned-qqp
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [6]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [ ]:
qqp_preprocessed["validation"]

Dataset({
    features: ['text1', 'text2', 'label', 'idx', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 40430
})

In [ ]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [ ]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [ ]:
from tqdm import trange, tqdm

BATCH_SIZE = 32
device = "cuda"
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=transformers.default_data_collator, num_workers=4
)
model = model.to(device)

def test_model(model, val_loader):
    correct_samples = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader):
            predicted = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                token_type_ids=batch["token_type_ids"].to(device),
            )
            preds = torch.argmax(predicted.logits, dim=-1).cpu()
            correct_samples += (batch["labels"]==preds).sum()

    accuracy = correct_samples / val_set.num_rows
    return accuracy


100%|██████████| 1264/1264 [01:15<00:00, 16.77it/s]

tensor(36726.)


In [ ]:
print(accuracy)


tensor(0.9084)


In [ ]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

In [33]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorForTokenClassification
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased" ,
)
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased" ,
    num_labels=2,  # QQP
    dtype=torch.float32,
)

model = AutoModelForSequenceClassification.from_pretrained(r"D:\stuff\Study\YSDA\nlp_course\week04_transfer\trainer_output\checkpoint-22500")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [36]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )
    result["label"] = examples["label"]
    return result

qqp = datasets.load_dataset("SetFit/qqp")
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Repo card metadata block was not found. Setting CardData to empty.


In [37]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="binary"),
    }

In [38]:
# Load model directly
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    eval_strategy="steps",
    eval_steps=2000,
    logging_steps=2000,
    output_dir=None,
    report_to="none",

    learning_rate=1e-6,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=qqp_preprocessed["train"],
    eval_dataset=qqp_preprocessed["validation"],
    compute_metrics=compute_metrics
    )


In [ ]:
trainer.train()

In [40]:
from tqdm import trange, tqdm

BATCH_SIZE = 32
device = "cuda"

def test_model(model, dataset):
    correct_samples = 0.0

    data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=32, shuffle=False, collate_fn=transformers.default_data_collator, num_workers=4)
    model = model.to(device)
    with torch.no_grad():
        for batch in tqdm(data_loader):
            predicted = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                token_type_ids=batch["token_type_ids"].to(device),
            )
            preds = torch.argmax(predicted.logits, dim=-1).cpu()
            correct_samples += (batch["labels"]==preds).sum()

    accuracy = correct_samples / dataset.num_rows
    return accuracy


test_model(model, qqp_preprocessed["validation"])



100%|██████████| 1264/1264 [00:44<00:00, 28.63it/s]


tensor(0.9015)

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [3]:
sentences = [
    "The small cat sat comfortably on the soft mat near the window.",
    "A small cat sat comfortably on the soft mat near the window.",
    "The fluffy cat was sitting on the mat by the window.",
    "A fluffy cat was sitting on the mat by the window.",
    "The cat decided to rest on the warm mat in the living room.",
    "A cat decided to rest on the warm mat in the living room.",
    "The cat stretched out and slept on the mat all afternoon.",
    "A cat stretched out and slept on the mat all afternoon.",
    "The dog ran quickly through the green park chasing a ball.",
    "A dog ran quickly through the green park chasing a ball.",
    "The excited dog ran through the park chasing after children.",
    "An excited dog ran through the park chasing after children.",
    "The dog loves to run and play in the park daily.",
    "A dog loves to run and play in the park daily.",
    "She drinks a cup of coffee every single morning without fail.",
    "She drinks her coffee every morning before starting work.",
    "She enjoys drinking coffee each morning while reading news.",
    "She drinks coffee every morning to help wake up.",
    "She always drinks coffee in the morning after waking up.",
    "She drinks coffee daily as part of her morning routine.",
    "The bright sun is shining intensely in the clear blue sky.",
    "The sun shines brightly overhead on this beautiful summer day.",
    "The sun is shining and warming everything beneath its rays.",
    "Today the sun is bright and there are no clouds.",
    "The sun shines down on the city from early morning.",
    "The sun is shining brightly making the temperature rise quickly.",
    "The children play outside in the backyard during the afternoon.",
    "Children play outside happily when the weather is nice.",
    "The kids play outside together after finishing their homework.",
    "Kids play outside in the yard until the sun sets.",
    "The children are playing outside with their friends and toys.",
    "Children are playing outside enjoying the warm summer weather.",
    "The kids are playing outside running around and having fun.",
    "Kids are playing outside in the fresh air today.",
    "Outside in the garden the children play games together.",
    "Outside children play while their parents watch from nearby.",
    "The woman purchased fresh vegetables from the local market yesterday.",
    "A woman purchased fresh vegetables from the local market yesterday.",
    "She bought fresh vegetables at the farmer's market downtown.",
    "She purchased fresh vegetables from the market for dinner.",
    "The man walked his dog along the quiet neighborhood street.",
    "A man walked his dog along the quiet neighborhood street.",
    "He walked his dog early in the morning before work.",
    "He walked his dog through the neighborhood every evening.",
    "The students studied hard in the library for their exams.",
    "Students studied hard in the library preparing for finals.",
    "They studied together in the library for several hours.",
    "The group studied in the library until it closed.",
    "She studied at the library for her upcoming test.",
    "She studied hard in the library all week long."
]

In [4]:
import datasets


def find_duplicates(sentences, model, k=5):
    BATCH_SIZE
    pairs =   {"text1" : [], "text2" : []}  
    for i, text1 in enumerate(sentences[:-1]):
        for text2 in sentences[i+1:]:
             pairs["text1"].append(text1)
             pairs["text2"].append(text2)
    inputs = tokenizer(
      pairs["text1"], 
      pairs["text2"],     
      padding="max_length",
      max_length=MAX_LENGTH, 
      return_tensors="pt"
    )

    device = "cuda"
    model = model.to(device)
    with torch.no_grad():
        #for ind, batch in enumerate(ds):
        predicted = model(
            input_ids=torch.tensor(inputs["input_ids"]).to(device),
            attention_mask=torch.tensor(inputs["attention_mask"]).to(device),
            token_type_ids=torch.tensor(inputs["token_type_ids"]).to(device),
        )
        print(predicted.logits)
        labels = torch.softmax(predicted.logits, dim=-1).cpu()[:, 1]
    pairs["dupe_prob"] = labels
    return pairs

pairs = find_duplicates(sentences, model)

NameError: name 'model' is not defined

In [94]:
for t1, t2, p in zip(pairs["text1"], pairs["text2"], pairs["dupe_prob"]):
    if p >= 0.9:
        print(t1, t2)


The fluffy cat was sitting on the mat by the window. A fluffy cat was sitting on the mat by the window.
The cat decided to rest on the warm mat in the living room. A cat decided to rest on the warm mat in the living room.
The cat stretched out and slept on the mat all afternoon. A cat stretched out and slept on the mat all afternoon.
The dog ran quickly through the green park chasing a ball. A dog ran quickly through the green park chasing a ball.
The excited dog ran through the park chasing after children. An excited dog ran through the park chasing after children.
The dog loves to run and play in the park daily. A dog loves to run and play in the park daily.
She drinks a cup of coffee every single morning without fail. She drinks coffee daily as part of her morning routine.
She drinks coffee every morning to help wake up. She always drinks coffee in the morning after waking up.
Children play outside happily when the weather is nice. Children are playing outside enjoying the warm summ

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>